In [ ]:
#seq2seq英译法案例
#案例步骤：
#1.数据预处理：读取数据，构建词汇表，转换为索引
#2.模型定义：定义编码器和解码器
#3.训练模型：定义损失函数和优化器，训练模型
#4.评估模型：测试模型性能，进行翻译

import re #正则表达式模块
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import time

import random #随机数生成器
import matplotlib.pyplot as plt

device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #设备适配
SOS_token=0 #设置开始标记和结束标记
EOS_token=1

max_length=10 #设置最大句子长度
path='./data/eng-fra-v2.txt'




In [ ]:
#数据预处理
#读取文档数据，构建词汇表和索引映射
eng_sentences=[] #英语句子列表
fra_sentences=[]
with open(path,'r',encoding='utf-8') as f:
    sentences=f.read().strip().split('\n')
    print(sentences[:5]) #打印前5行数据
    for sentence in sentences:
        eng,fra=sentence.split('\t')
        eng_sentences.append(eng)
        fra_sentences.append(fra)
    print(eng_sentences[:5])
    print(len(fra_sentences))

#构建两个句子列表的字典对应
eng_fra_dict={}
for i in range(len(eng_sentences)):
    eng_fra_dict.update({eng_sentences[i]:fra_sentences[i]})
# print(eng_fra_dict)

#构建词汇表和索引映射
eng_words=[]
fra_words=[]
for sentence in eng_sentences:
    eng_words.extend(sentence.split(' '))
print(eng_words[:5])

for sentence in fra_sentences:
    fra_words.extend(sentence.split(' '))
print(fra_words[:5])

eng_vocab=list(set(eng_words)) #英语词汇表
fra_vocab=list(set(fra_words))
print(len(eng_vocab))
print(len(fra_vocab))

eng_word_idx={'SOS':SOS_token,'EOS':EOS_token} #英语词汇索引映射
fra_word_idx={'SOS':SOS_token,'EOS':EOS_token} #法语词汇
i=2
j=2
for word in eng_vocab:
    eng_word_idx[word]=i
    i+=1
for word in fra_vocab:
    fra_word_idx[word]=j
    j+=1
# print(fra_word_idx)
# print(fra_word_idx.items())

eng_idx_word={i:w for w,i in eng_word_idx.items()}
fra_idx_word={i:w for w,i in fra_word_idx.items()}
print(eng_idx_word)

eng_seq_idx=[]
fra_seq_idx=[]
#将句子转换为索引序列
for sentence in eng_sentences:
    tem=[eng_word_idx[w] for w in sentence.split(' ')]
    eng_seq_idx.append(tem)
print(eng_seq_idx[:5])

for sentence in fra_sentences:
    tem=[fra_word_idx[w] for w in sentence.split(' ')]
    fra_seq_idx.append(tem)
print(fra_seq_idx[:5])

In [ ]:
#构建数据集
eng_fra_list=list(eng_fra_dict.items())

#定义数据集类
class SeqDataset(Dataset):
    def __init__(self,eng_fra_list):
        self.eng_fra_pairs=eng_fra_list
        self.length=len(eng_fra_list)
    def __len__(self):
        return self.length
    def __getitem__(self,idx):
        eng=self.eng_fra_pairs[idx][0]
        fra=self.eng_fra_pairs[idx][1]
        eng_idx=[eng_word_idx[w] for w in eng.split(' ')]
        eng_idx.append(eng_word_idx['EOS']) #添加结束标记
        fra_idx=[fra_word_idx[w] for w in fra.split(' ')]
        fra_idx.append(fra_word_idx['EOS'])
        x=torch.tensor(eng_idx)
        y=torch.tensor(fra_idx)
        return x,y

#实例化数据集和数据加载器
dataset=SeqDataset(eng_fra_list)
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)
for x,y in dataloader:
    print(x.shape)
    print(y.shape)


In [ ]:
#定义编码器和解码器
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(eng_word_idx),128) #词嵌入层
        self.gru=nn.GRU(128,256,1,batch_first=True)
    def forward(self,x,h0):
        x=self.embed(x)
        output,hn=self.gru(x,h0)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)

#检验编码器
encoder=Encoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0=encoder.init_hidden(x.size(0))
    output,hn=encoder(x,h0)
    print(output.shape)
    print(hn.shape)


#这里output的shape是(batch_size,seq_len,hidden_size)，hn的shape是(num_layers*num_directions,batch_size,hidden_size)


In [ ]:
#定义解码器
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(fra_word_idx),128)
        self.gru=nn.GRU(128,256,1,batch_first=True)
        self.fc1=nn.Linear(256,len(fra_word_idx))
    def forward(self,x,h0):
        x=self.embed(x)
        x=F.relu(x) #relu激活函数，防止梯度消失，过拟合
        output,hn=self.gru(x,h0)
        output=self.fc1(output)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)
#检验解码器
decoder=Decoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0=decoder.init_hidden(x.size(0))
    output,hn=decoder(x,h0)
    # print(output.shape)
    # print(hn.shape)
    print(y.shape)



In [13]:
#添加attention机制
#Q:解码器当前隐藏状态
#K:编码器输出的所有隐藏状态 这里是output
#V和K相同
#上下文和输入拼接后输入解码器
class AttentionDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed=nn.Embedding(len(fra_word_idx),128)
        self.gru=nn.GRU(128+256,256,1,batch_first=True) #输入维度是输入维度加上上下文维度
        self.fc1=nn.Linear(256,len(fra_word_idx))
        self.fc2=nn.Linear(256,256)

    def forward(self,x,h0,encoder_outputs):
        x=self.embed(x)
        x=nn.Dropout(0.3)(x) #添加dropout层，防止过拟合
        #Q:h0
        #K:encoder_outputs
        #V:encoder_outputs
        encoder_outputs=self.fc2(encoder_outputs) #将编码器输出映射到解码器隐藏状态维度
        weights=torch.bmm(h0,encoder_outputs.transpose(2,1))/torch.sqrt(torch.tensor(128))
        weights=F.softmax(weights,dim=1)
        context=torch.bmm(weights,encoder_outputs) #计算上下文向量

        context_expanded=context.expand(-1,x.size(1),-1) #将上下文向量扩展到与输入相同的时间步长
        decoder_input=torch.cat((context_expanded,x),dim=2)
        output,hn=self.gru(decoder_input,h0)
        output=self.fc1(output)
        return output,hn
    def init_hidden(self,batch_size):
        return torch.zeros(1,batch_size,256,device=device)


In [ ]:


#验证解码器
attention_decoder=AttentionDecoder().to(device)
for x,y in dataloader:
    x=x.to(device)
    y=y.to(device)
    h0_decoder=attention_decoder.init_hidden(x.size(0))
    h0_encoder=encoder.init_hidden(x.size(0))
    encoder_output,hn=encoder(x,h0_encoder)
    output,hn=attention_decoder(x,h0_decoder,encoder_output)
    print(output.shape)
    print(hn.shape)

In [40]:
#训练模型
encoder=Encoder().to(device)
decoder=AttentionDecoder().to(device)
criterion=nn.CrossEntropyLoss()
epochs=1
encoder_optimizer=optim.Adam(encoder.parameters(),lr=0.001)
decoder_optimizer=optim.Adam(decoder.parameters(),lr=0.001)
for epoch in range(epochs):
    encoder.train()
    decoder.train()
    total_samples,total_correct,total_loss,start=0,0,0,time.time()
    for x,y in dataloader:
        x=x.to(device)
        y=y.to(device)
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()
        h0_encoder=encoder.init_hidden(x.size(0))
        h0_decoder=decoder.init_hidden(x.size(0))
        encoder_output,hn_encoder=encoder(x,h0_encoder)
        output,hn_decoder=decoder(x,h0_decoder,encoder_output)
        loss=criterion(output.permute(0,2,1),y)
        loss.backward()
        encoder_optimizer.step()
        decoder_optimizer.step()
        total_loss+=loss.item()*x.size(0)
        total_samples+=x.size(0)
        total_correct+=(torch.argmax(output,dim=2)==y).sum()
    end=time.time()
    print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/total_samples:.4f}, Accuracy: {total_correct/total_samples:.4f}, Time: {end-start:.2f}s')







RuntimeError: Expected target size [1, 8], got [1, 10]